In [47]:
from aiomoex import get_board_candles
import asyncio
import aiohttp
from datetime import datetime
import pandas as pd

VALID_INTERVALS = {
    1: "1 минута",
    10: "10 минут", 
    60: "1 час",
    24: "1 день",
    7: "1 неделя",
    31: "1 месяц"
}

async def fetch_ticker_data(session: aiohttp.ClientSession, ticker: str, interval: int, start_date: str, end_date: str) -> dict[str, pd.DataFrame]:
    try:
        res = await get_board_candles(session, ticker, interval, start_date, end_date)
        if res:
            df = pd.DataFrame(res)
            
            # Убираем колонку 'end' если она существует
            if 'end' in df.columns:
                df = df.drop('end', axis=1)
            
            # Оставляем только дату (без времени), если это дневные данные или выше
            if interval in [24, 7, 31]:
                df['begin'] = pd.to_datetime(df['begin']).dt.date
            
            # Перемещаем дату в первую колонку
            if 'begin' in df.columns:
                cols = ['begin'] + [col for col in df.columns if col != 'begin']
                df = df[cols]
            
            df['ticker'] = ticker
            return {ticker: df}
        else:
            return {ticker: pd.DataFrame()}
    except Exception as e:
        print(f'Ошибка парсинга. Не удалось получить данные для {ticker}, {e}')
        return {ticker: pd.DataFrame()}

async def get_moex_data(tickers: list[str], start_date: str, end_date: str, interval: int = 24) -> dict[str, pd.DataFrame]:
    if interval not in VALID_INTERVALS:
        raise ValueError(f"Неверный интервал. Допустимые значения: {list(VALID_INTERVALS.keys())}")
    
    async with aiohttp.ClientSession() as session:
        coros = [fetch_ticker_data(session, ticker, interval, start_date, end_date) for ticker in tickers]
        stock_data = await asyncio.gather(*coros)
    
    stock_data_dict = {k: v for d in stock_data for k, v in d.items()}
    return stock_data_dict

In [53]:
stock_data = await get_moex_data(
    tickers=['SBER', 'GAZP'],
    start_date='2022-02-18', 
    end_date='2025-10-01', 
    interval=24
)

In [55]:
stock_data['SBER'].iloc[:10]

,begin,open,close,high,low,value,volume,ticker
0,2022-02-18,262.13,250.28,267.00,245.00,6.903378e+10,270594740,SBER
1,2022-02-21,249.15,201.00,258.32,184.30,2.385837e+11,1084508440,SBER
2,2022-02-22,198.72,208.53,219.90,182.03,1.886633e+11,946020570,SBER
3,2022-02-24,187.54,132.18,187.54,89.59,9.828140e+10,829355880,SBER
4,2022-02-25,123.75,131.12,152.89,115.11,5.176981e+10,396287150,SBER
5,2022-03-24,131.00,136.24,156.20,130.15,2.269207e+10,159464350,SBER
6,2022-03-25,137.00,131.50,147.00,128.20,7.725368e+09,57662950,SBER
7,2022-03-28,130.60,125.00,131.47,125.00,4.203875e+09,33212430,SBER
8,2022-03-29,126.16,128.77,137.57,122.00,9.470337e+09,72338740,SBER
9,2022-03-30,136.89,134.60,138.40,131.11,4.754054e+09,35675450,SBER
